In [48]:
import numpy as np
import h5py, glob
import matplotlib.pyplot as plt
import pandas as pd
from scipy.stats import ks_2samp
from pandas.plotting import table 
IT73_quality_cuts = []# ['IceTop_reco_succeeded', 'Laputop_FractionContainment'] #pick which Quality Cuts you want to apply
run_years = np.arange(2011, 2022, 1)
path_to_arrays = '/data/ana/CosmicRay/Anisotropy/IceTop/ITpass2/output/icecube/s125arrays' #update to correct directory

#determining station cuts
#for 2021 station cuts will be tier3_min_station=<tier3 nstations<tier4_min_station and tier4_min_station=<tier4 nstations
#for 2011 station cuts will be tier2_min_station=<tier2 nstations<(tier3_min_station + (1/years_to_increase * 11))....

tier2_min_station = 5 #this is the tier2 energy minimum cut for 2011
tier3_min_station_2021 = 5 #this is the tier2 energy minimum cut for 2021
tier4_min_station_2021 = 12 #this is the tier4 energy minimum cut for 2021
years_to_increase_cut = 2 #for example, every 2 years cut increases by 1, so 2021 tier 3 is 5-12, but 2019 tier 3 is 6-13

In [49]:
#this block loads the s125, stations, and zenith arrays that were created above
for run_year in run_years:
    globals()['s125_'+str(run_year)] = np.load('{}/s125_{}.npy'.format(path_to_arrays, run_year), 'r')
    globals()['stations_'+str(run_year)] = np.load('{}/stations_{}.npy'.format(path_to_arrays, run_year), 'r')
    globals()['zenith_' + str(run_year)] = np.load('{}/zenith_{}.npy'.format(path_to_arrays, run_year), 'r')
   # print(len(globals()['s125_'+str(run_year)]), len(globals()['stations_'+str(run_year)]), len(globals()['zenith_' + str(run_year)]))

In [50]:
#this block applies the IT73 quality cuts that you input at the top
for run_year in run_years:
    finalcut = np.ones(len(globals()['s125_'+str(run_year)]), dtype=int)
    for IT73_quality_cut in IT73_quality_cuts:
        globals()[IT73_quality_cut+'_cut'] = np.load('{}/{}_{}.npy'.format(path_to_arrays, IT73_quality_cut, str(run_year)))
        for i in range(len(globals()[IT73_quality_cut+'_cut'])):
            if globals()[IT73_quality_cut+'_cut'][i] == 0:
                finalcut[i] = 0
    globals()['s125_'+str(run_year)] = globals()['s125_'+str(run_year)][np.where(finalcut==1)]
    globals()['stations_'+str(run_year)] = globals()['stations_'+str(run_year)][np.where(finalcut==1)]
    globals()['zenith_'+str(run_year)] = globals()['zenith_'+str(run_year)][np.where(finalcut==1)]

In [51]:
#eliminating nan and inf events, and applying the zenith quality cut
for run_year in run_years:
    zenith_cut = np.deg2rad(55)
    #print(run_year, len(globals()['s125_'+str(run_year)]), len(globals()['stations_'+str(run_year)]))
    globals()['stations_'+str(run_year)] = globals()['stations_'+str(run_year)][np.where(~np.isnan(globals()['s125_'+str(run_year)]))]
    globals()['zenith_'+str(run_year)] = globals()['zenith_'+str(run_year)][np.where(~np.isnan(globals()['s125_'+str(run_year)]))]
    globals()['s125_'+str(run_year)] = globals()['s125_'+str(run_year)][np.where(~np.isnan(globals()['s125_'+str(run_year)]))]
    
    globals()['stations_'+str(run_year)] = globals()['stations_'+str(run_year)][np.where(np.isfinite(globals()['s125_'+str(run_year)]))]
    globals()['zenith_'+str(run_year)] = globals()['zenith_'+str(run_year)][np.where(np.isfinite(globals()['s125_'+str(run_year)]))]
    globals()['s125_'+str(run_year)] = globals()['s125_'+str(run_year)][np.where(np.isfinite(globals()['s125_'+str(run_year)]))]
    
    globals()['s125_'+str(run_year)] = globals()['s125_'+str(run_year)][np.where(globals()['zenith_' + str(run_year)] < zenith_cut)]
    globals()['stations_'+str(run_year)] = globals()['stations_'+str(run_year)][np.where(globals()['zenith_' + str(run_year)] < zenith_cut)]
    #print(run_year, len(globals()['s125_'+str(run_year)]), len(globals()['stations_'+str(run_year)]))
 

In [52]:
#this block applys the nstation cuts to create the different energy tiers
cuts = []

reversed_run_years = reversed(run_years)
i = 0
for run_year in reversed_run_years:
    if run_year == 2021:
        tier3_min_station = tier3_min_station_2021
        tier4_min_station = tier4_min_station_2021
    if i == years_to_increase_cut: #this if statement increases the station cut boundries going back in the increment specified by years_to_increase
        tier3_min_station +=1
        tier4_min_station += 1
        i = 0
    globals()['tier4'+str(run_year)] = globals()['s125_'+str(run_year)][np.where(globals()['stations_'+str(run_year)] >= tier4_min_station)]
    globals()['tier3'+str(run_year)] = globals()['s125_'+str(run_year)][np.where((globals()['stations_'+str(run_year)] >= tier3_min_station) & (globals()['stations_'+str(run_year)] < tier4_min_station))]
      
    i += 1
    
    if run_year in [2011, 2012, 2013, 2014, 2015]:
        globals()['tier2'+str(run_year)] = globals()['s125_'+str(run_year)][np.where((globals()['stations_'+str(run_year)] >= tier2_min_station) & (globals()['stations_'+str(run_year)] < tier3_min_station))]
        display_tier2_cuts = '{} ≤ bins < {}'.format(tier2_min_station, tier3_min_station)
    else:
        globals()['tier2'+str(run_year)] = 0
        display_tier2_cuts =''
    display_tier3_cuts = '{} ≤ bins < {}'.format(tier3_min_station, tier4_min_station) #this is for displaying the station cuts on the table
    display_tier4_cuts = 'bins ≥ {}'.format(tier4_min_station) 
    cuts.append([display_tier2_cuts, display_tier3_cuts, display_tier4_cuts])

cuts = reversed(cuts)
tab = dict(zip(run_years, cuts))   
df = pd.DataFrame.from_dict(tab, orient='index', columns = pd.MultiIndex.from_product([['Stations Cuts'], ['Tier 2', 'Tier 3', 'Tier 4']]))
display(df) #this is a prettier way to display the table when using jupyter


Stations Cuts                           
             Tier 2          Tier 3     Tier 4
2011  5 ≤ bins < 10  10 ≤ bins < 17  bins ≥ 17
2012   5 ≤ bins < 9   9 ≤ bins < 16  bins ≥ 16
2013   5 ≤ bins < 9   9 ≤ bins < 16  bins ≥ 16
2014   5 ≤ bins < 8   8 ≤ bins < 15  bins ≥ 15
2015   5 ≤ bins < 8   8 ≤ bins < 15  bins ≥ 15
2016                  7 ≤ bins < 14  bins ≥ 14
2017                  7 ≤ bins < 14  bins ≥ 14
2018                  6 ≤ bins < 13  bins ≥ 13
2019                  6 ≤ bins < 13  bins ≥ 13
2020                  5 ≤ bins < 12  bins ≥ 12
2021                  5 ≤ bins < 12  bins ≥ 12

"\nfig, ax = plt.subplots(figsize= (12, 15)) # no visible frame\nax.xaxis.set_visible(False)  # hide the x axis\nax.yaxis.set_visible(False)  # hide the y axis\nax.set(frame_on = False)\n\ntable(ax, df, loc='center')  # where df is your data frame\n\nfig.savefig('stations.png', bbox_inches='tight')\n"